In [39]:
from ultralytics import YOLO
import cv2
import numpy as np
import math
import torch

# ============================================================
# PATHS
# ============================================================

SPOON_MODEL_PATH = (
    r"C:\Study\Major\runs\detect\runs\spoon_yolo11m\weights\last.pt"
)

FOOD_MODEL_PATH = (
    r"C:\Study\Major\Food\classif.pt"
)

IMAGE_PATH = r"C:\Study\Major\Spoon\Tests\Test3.jpeg"

SAM_MODEL_PATH = r"C:\Study\Major\sam2_b.pt"
IMAGE_PATH = r"Spoon\Tests\Test3.jpeg"

# SAM_MODEL = "sam3.pt"
MIDAS_MODEL = "DPT_Hybrid"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# LOAD MODELS
# ============================================================

print("Loading models...")

# spoon_model = YOLO(SPOON_MODEL_PATH)

# food_model = YOLO(FOOD_MODEL_PATH)

from ultralytics import SAM

# sam_model = SAM(SAM_MODEL_PATH)

print("Models loaded.")

Loading models...
Models loaded.


In [3]:
from Pipeline.Spoon_detection import SpoonDetection
SpoonDetection = SpoonDetection(SPOON_MODEL_PATH)
spoon_est = SpoonDetection.detect_spoon(IMAGE_PATH)

Loading models...

image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 640x608 1 Spoon, 21.7ms
Speed: 41.2ms preprocess, 21.7ms inference, 25.4ms postprocess per image at shape (1, 3, 640, 608)


In [4]:
spoon_est

{'spoon_length_px': 1021.75341796875,
 'spoon_height_px': 732.7967834472656,
 'spoon_width_px': 1021.75341796875,
 'spoon_confidence': 0.947821855545044}

In [5]:
result = spoon_est.copy()

In [6]:
from Pipeline.Food_detection import FoodDetection
FoodDetection = FoodDetection(FOOD_MODEL_PATH)

food_boxes = FoodDetection.detect_food(IMAGE_PATH)


image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 512x480 1 poha, 19.6ms
Speed: 4.1ms preprocess, 19.6ms inference, 2.4ms postprocess per image at shape (1, 3, 512, 480)


In [7]:
food_boxes

[{'x1': 62.79640197753906,
  'y1': 451.69561767578125,
  'x2': 1019.7351684570312,
  'y2': 1359.32275390625,
  'class_id': 45,
  'name': 'poha',
  'confidence': 0.9494404196739197}]

In [8]:
from Pipeline.SAMMasking import SAM2
masking_model = SAM2(SAM_MODEL_PATH)


Running SAM 2...


In [9]:
result["food_boxes"] = food_boxes

In [10]:
final = masking_model.detect_food_area(IMAGE_PATH, result)


Segmenting: poha
Mask area: 558,605 px
Mask/BBox ratio: 0.643
Food/Spoon area ratio: 0.746


In [11]:

image = cv2.imread(IMAGE_PATH)

In [12]:


# ============================================================
# VISUALIZATION
# ============================================================

output = image.copy()
for food in final:

    mask = food["mask"]
    colored_mask = np.zeros_like(
        output
    )

    colored_mask[:, :, 1] = (
        mask * 255
    )


    output = cv2.addWeighted(
        output,
        1.0,
        colored_mask,
        0.35,
        0
    )

    x1, y1, x2, y2 = food["bbox"]
    cv2.rectangle(
        output,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )
    label = (
        f"{food['name']} "
        f"{food['confidence']:.2f}"
    )
    cv2.putText(
        output,
        label,
        (x1, max(30, y1 - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )
cv2.imshow(
    "Food Segmentation",
    output
)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [14]:
SPOON_LENGTH_CM = 13.5

pixels_per_cm = (
    result["spoon_length_px"] / SPOON_LENGTH_CM
)

cm_per_pixel = 1 / pixels_per_cm
cm_per_pixel

0.013212581198737808

In [15]:
for food in final:
    segmented_area_cm2 = food["mask_area_px"] * (cm_per_pixel ** 2)
    print(f"Segmented area: {segmented_area_cm2:.2f} cm²")

Segmented area: 97.52 cm²


In [17]:
from Metrics.Quantities import FOOD_QUANTITY

In [18]:
for food in final:
    food_name = food["name"]
    baseline = FOOD_QUANTITY.get(food_name)
    if baseline["mode"] == "area":
        segmented_area_cm2 = food["mask_area_px"] * (cm_per_pixel ** 2)
        estimated_weight_g = segmented_area_cm2 * baseline["g_per_cm2"]
        print(f"{food_name}: {estimated_weight_g:.2f} g")
    elif baseline["mode"] == "count":
        segmented_area_cm2 = food["mask_area_px"] * (cm_per_pixel ** 2)
        estimated_weight_g = (segmented_area_cm2 / baseline["baseline_area_cm2"]) * baseline["baseline_g"]
        print(f"{food_name}: {estimated_weight_g:.2f} g")
    elif baseline["mode"] == "liquid":
        segmented_area_cm2 = food["mask_area_px"] * (cm_per_pixel ** 2)
        estimated_volume_ml = segmented_area_cm2 * baseline["ml_per_cm2"]
        print(f"{food_name}: {estimated_volume_ml:.2f} ml")




poha: 112.14 g


In [25]:
from ultralytics.models.sam import SAM3SemanticPredictor

overrides = {
    "conf": 0.25,
    "task": "segment",
    "mode": "predict",
    "model": r"C:\Study\Major\sam3.pt",
    "quantize": 16,
    "save": False,
    "save_txt": False,
}

predictor = SAM3SemanticPredictor(overrides=overrides)

predictor.set_image(IMAGE_PATH)

results = predictor(
    text=["spoon"]
)

# Results are only kept in memory
print(results)

Ultralytics 8.4.118  Python-3.14.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)


WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Study\Major\Spoon\Tests\Test3.jpeg: 644x644 1 spoon, 169.5ms
Speed: 7.2ms preprocess, 169.5ms inference, 3.4ms postprocess per image at shape (1, 3, 644, 644)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
depth: None
keypoints: None
masks: ultralytics.engine.results.Masks object
names: ['spoon']
obb: None
orig_img: array([[[ 38,  40,  48],
        [ 38,  40,  48],
        [ 37,  38,  48],
        ...,
        [132, 135, 139],
        [133, 136, 140],
        [133, 136, 140]],

       [[ 35,  37,  45],
        [ 35,  37,  45],
        [ 34,  35,  45],
        ...,
        [131, 134, 138],
        [133, 136, 140],
        [133, 136, 140]],

       [[ 34,  36,  44],
        [ 34,  36,  44],
        [ 34,  36,  44],
        ...,
        [130, 133, 137],
        [132, 135, 139],
        [133, 136, 140]],

       ...,

       [[ 45,  45,  51],
 

In [24]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: ['spoon']
 obb: None
 orig_img: array([[[ 38,  40,  48],
         [ 38,  40,  48],
         [ 37,  38,  48],
         ...,
         [132, 135, 139],
         [133, 136, 140],
         [133, 136, 140]],
 
        [[ 35,  37,  45],
         [ 35,  37,  45],
         [ 34,  35,  45],
         ...,
         [131, 134, 138],
         [133, 136, 140],
         [133, 136, 140]],
 
        [[ 34,  36,  44],
         [ 34,  36,  44],
         [ 34,  36,  44],
         ...,
         [130, 133, 137],
         [132, 135, 139],
         [133, 136, 140]],
 
        ...,
 
        [[ 45,  45,  51],
         [ 67,  67,  73],
         [ 83,  83,  89],
         ...,
         [ 86,  86,  92],
         [ 93,  93,  99],
         [ 99,  99, 105]],
 
        [[ 87,  87,  93],
         [103, 103, 109],
         [109,

In [33]:
overrides

{'conf': 0.25,
 'task': 'segment',
 'mode': 'predict',
 'model': 'C:\\Study\\Major\\sam3.pt',
 'quantize': 16,
 'save': False,
 'save_txt': False,
 'batch': 1}

In [35]:
overrides2 = {
    'conf': 0.25,
 'task': 'segment',
 'mode': 'predict',
 'model': 'C:\\Study\\Major\\sam3.pt',
 'quantize': 16,
 
 'batch': 1
 }

In [38]:
from ultralytics.models.sam import SAM3SemanticPredictor
bowl_model = SAM3SemanticPredictor(overrides=overrides2)

result = bowl_model.set_image(r"C:\Study\Major\Spoon\BowlTest1.jpg")
result = bowl_model(
    text=["bowl", "food container"]
)

Ultralytics 8.4.118  Python-3.14.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)


WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Study\Major\Spoon\BowlTest1.jpg: 644x644 2 bowls, 2 food containers, 538.9ms
Speed: 12.8ms preprocess, 538.9ms inference, 7.2ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Study\Major\Working\runs\segment\predict-2


In [2]:
import cv2
import numpy as np

from Food_detection import FoodDetection
from Spoon_detection import SpoonDetection
from FoodAreap import FoodSegmenter
from SpoonLength import SpoonDepthEstimator


class FoodAreaPipeline:
    """
    Pipeline:
    1. Detect food type.
    2. Detect spoon/fork.
    3. Segment the detected utensil.
    4. Get Depth Pro depth map.
    5. Calibrate utensil length to 15 cm and create a
       depth-adjusted cm/pixel map.
    6. Segment each food and estimate its visible area in cm².

    The depth-adjusted area is an approximation. It does not
    estimate food volume or mass.
    """

    UTENSIL_LENGTH_CM = 15.0

    def __init__(
        self,
        food_model_path,
        utensil_model_path,
        sam_model_path=r"C:\Study\Major\sam3.pt",
        depth_api_url="http://localhost:8000/predict-depth-map",
        food_confidence=0.50,
        utensil_confidence=0.25,
    ):
        print("\nInitializing food area pipeline...")

        # Stage 1: food detector
        self.food_detector = FoodDetection(
            food_model_path
        )

        # Stage 2: spoon/fork detector
        self.utensil_detector = SpoonDetection(
            utensil_model_path
        )

        # FoodSegmenter loads YOLO + SAM 3.
        # Its SAM predictor will also be reused for utensil masks.
        self.food_segmenter = FoodSegmenter(
            yolo_model_path=food_model_path,
            sam_model_path=sam_model_path,
            conf=food_confidence,
        )

        # Reuse the existing Depth Pro FastAPI client.
        # This class also loads a SAM model internally, although
        # this pipeline only uses its API method.
        self.depth_estimator = SpoonDepthEstimator(
            sam_model_path=sam_model_path,
            depth_api_url=depth_api_url,
        )

        self.utensil_confidence = utensil_confidence

        print("Pipeline initialized.")

    # ---------------------------------------------------------
    # UTILITY: NORMALIZE A SAM MASK TO ORIGINAL IMAGE SIZE
    # ---------------------------------------------------------
    @staticmethod
    def _resize_mask(mask, image_shape):
        height, width = image_shape[:2]

        if mask.shape != (height, width):
            mask = cv2.resize(
                mask.astype(np.uint8),
                (width, height),
                interpolation=cv2.INTER_NEAREST,
            ).astype(bool)

        return mask.astype(bool)

    # ---------------------------------------------------------
    # UTILITY: SELECT THE MASK THAT BEST MATCHES A DETECTION
    # ---------------------------------------------------------
    @classmethod
    def _select_mask_for_box(
        cls,
        masks,
        bbox,
        image_shape,
    ):
        """
        Select the SAM mask with the greatest fraction of its
        pixels inside the detector's bounding box.
        """
        x1, y1, x2, y2 = map(int, bbox)

        height, width = image_shape[:2]

        x1 = max(0, min(x1, width - 1))
        x2 = max(x1 + 1, min(x2, width))
        y1 = max(0, min(y1, height - 1))
        y2 = max(y1 + 1, min(y2, height))

        best_mask = None
        best_score = -1.0

        for raw_mask in masks:
            mask = cls._resize_mask(
                raw_mask,
                image_shape,
            )

            mask_area = int(mask.sum())

            if mask_area == 0:
                continue

            inside_box = mask[y1:y2, x1:x2]
            overlap_pixels = int(inside_box.sum())

            # Prefer masks that substantially overlap the
            # corresponding detection.
            score = overlap_pixels / mask_area

            if score > best_score:
                best_score = score
                best_mask = mask

        if best_mask is None or best_score <= 0:
            raise RuntimeError(
                "Could not find a SAM mask overlapping "
                "the detection bounding box."
            )

        return best_mask

    # ---------------------------------------------------------
    # STAGE 1: DETECT FOOD TYPE
    # ---------------------------------------------------------
    def detect_food(self, image_path):
        print("\n[STAGE 1] Detecting food types...")

        foods = self.food_detector.detect_food(
            image_path
        )

        for i, food in enumerate(foods):
            print(
                f"  Food {i + 1}: {food['name']} "
                f"(confidence={food['confidence']:.3f})"
            )

        return foods

    # ---------------------------------------------------------
    # STAGE 2: DETECT SPOON OR FORK
    # ---------------------------------------------------------
    def detect_utensil(self, image_path):
        print("\n[STAGE 2] Detecting spoon/fork...")

        model = self.utensil_detector.model
        results = model(image_path)

        detections = []

        # Supports a YOLO model trained with classes named
        # "spoon" and/or "fork".
        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                confidence = float(box.conf[0])

                if confidence < self.utensil_confidence:
                    continue

                class_id = int(box.cls[0])
                names = model.names

                if isinstance(names, dict):
                    name = str(names[class_id])
                else:
                    name = str(names[class_id])

                normalized_name = name.lower().strip()

                if (
                    "spoon" not in normalized_name
                    and "fork" not in normalized_name
                ):
                    continue

                x1, y1, x2, y2 = map(
                    float,
                    box.xyxy[0].tolist(),
                )

                detections.append({
                    "name": normalized_name,
                    "class_id": class_id,
                    "confidence": confidence,
                    "bbox": (x1, y1, x2, y2),
                })

        if not detections:
            raise RuntimeError(
                "No spoon/fork detected. Check that your "
                "utensil YOLO model has spoon/fork classes."
            )

        # Use the highest-confidence detected utensil.
        utensil = max(
            detections,
            key=lambda item: item["confidence"],
        )

        print(
            f"  Utensil: {utensil['name']} "
            f"(confidence={utensil['confidence']:.3f})"
        )

        return utensil

    # ---------------------------------------------------------
    # STAGE 3: SEGMENT THE DETECTED UTENSIL USING SAM 3
    # ---------------------------------------------------------
    def segment_utensil(self, image_path, utensil):
        print("\n[STAGE 3] Segmenting utensil with SAM 3...")

        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(image_path)

        predictor = self.food_segmenter.sam_predictor

        predictor.set_image(image_path)

        # Prompt according to the YOLO-detected class.
        results = predictor(
            text=[utensil["name"]]
        )

        if not results or results[0].masks is None:
            raise RuntimeError(
                f"SAM 3 failed to segment {utensil['name']}."
            )

        raw_masks = (
            results[0].masks.data
            .detach()
            .cpu()
            .numpy()
        )

        mask = self._select_mask_for_box(
            masks=raw_masks,
            bbox=utensil["bbox"],
            image_shape=image.shape,
        )

        if int(mask.sum()) < 2:
            raise RuntimeError(
                "The utensil mask contains too few pixels."
            )

        # Find approximate long-axis endpoints using PCA.
        # Coordinates are stored as (x, y).
        ys, xs = np.where(mask)
        points = np.column_stack((xs, ys)).astype(
            np.float32
        )

        center = points.mean(axis=0)
        centered = points - center

        _, _, vh = np.linalg.svd(
            centered,
            full_matrices=False,
        )

        principal_axis = vh[0]
        projections = centered @ principal_axis

        point1 = points[np.argmin(projections)]
        point2 = points[np.argmax(projections)]

        utensil_length_px = float(
            np.linalg.norm(point2 - point1)
        )

        if utensil_length_px <= 0:
            raise RuntimeError(
                "Could not calculate utensil pixel length."
            )

        print(f"  Utensil mask pixels: {int(mask.sum())}")
        print(f"  Utensil length: {utensil_length_px:.2f} px")

        return {
            "mask": mask,
            "length_px": utensil_length_px,
            "endpoint1_xy": tuple(
                map(float, point1)
            ),
            "endpoint2_xy": tuple(
                map(float, point2)
            ),
        }

    # ---------------------------------------------------------
    # STAGE 4: GET DEPTH PRO DEPTH MAP
    # ---------------------------------------------------------
    def get_depth_map(self, image_path, image_shape):
        print("\n[STAGE 4] Requesting Depth Pro map...")

        # Existing SpoonDepthEstimator API returns the depth map
        # as a NumPy array.
        depth_map = self.depth_estimator.get_depth_from_api(
            image_path
        )

        height, width = image_shape[:2]

        if depth_map.ndim != 2:
            raise RuntimeError(
                f"Expected a 2D depth map; got {depth_map.shape}."
            )

        if depth_map.shape != (height, width):
            depth_map = cv2.resize(
                depth_map.astype(np.float32),
                (width, height),
                interpolation=cv2.INTER_LINEAR,
            )

        depth_map = depth_map.astype(np.float32)

        valid = np.isfinite(depth_map) & (depth_map > 0)

        if not valid.any():
            raise RuntimeError(
                "Depth Pro returned no valid positive depths."
            )

        print(f"  Depth map shape: {depth_map.shape}")
        print(
            f"  Valid depth range: "
            f"{depth_map[valid].min():.4f} - "
            f"{depth_map[valid].max():.4f}"
        )

        return depth_map

    # ---------------------------------------------------------
    # STAGE 5: CALIBRATE SCALE USING 15 CM UTENSIL
    # ---------------------------------------------------------
    def calculate_scale_map(
        self,
        depth_map,
        utensil_mask,
        utensil_length_px,
    ):
        print("\n[STAGE 5] Calculating depth-adjusted scale...")

        valid = np.isfinite(depth_map) & (depth_map > 0)

        utensil_depths = depth_map[
            utensil_mask & valid
        ]

        if utensil_depths.size == 0:
            raise RuntimeError(
                "No valid depth values inside utensil mask."
            )

        # Median is more robust than the minimum/maximum depth
        # to individual noisy pixels.
        reference_depth = float(
            np.median(utensil_depths)
        )

        # Calibrate the observed 2D utensil length to 15 cm
        # at its median depth.
        reference_cm_per_pixel = (
            self.UTENSIL_LENGTH_CM / utensil_length_px
        )

        # Approximate perspective adjustment:
        # pixel scale is proportional to depth.
        scale_map = np.full(
            depth_map.shape,
            np.nan,
            dtype=np.float32,
        )

        scale_map[valid] = (
            reference_cm_per_pixel
            * depth_map[valid]
            / reference_depth
        )

        print(
            f"  Assumed utensil length: "
            f"{self.UTENSIL_LENGTH_CM:.2f} cm"
        )
        print(
            f"  Reference depth: {reference_depth:.4f}"
        )
        print(
            f"  Reference scale: "
            f"{reference_cm_per_pixel:.6f} cm/pixel"
        )
        print(
            f"  Scale range: "
            f"{np.nanmin(scale_map):.6f} - "
            f"{np.nanmax(scale_map):.6f} cm/pixel"
        )

        return {
            "scale_map": scale_map,
            "reference_depth": reference_depth,
            "reference_cm_per_pixel": (
                reference_cm_per_pixel
            ),
        }

    # ---------------------------------------------------------
    # STAGE 6: SEGMENT EACH FOOD AND ESTIMATE ITS AREA
    # ---------------------------------------------------------
    def segment_food_item(
        self,
        image_path,
        food,
        image_shape,
    ):
        predictor = self.food_segmenter.sam_predictor

        # Use the detected food class as the SAM prompt.
        predictor.set_image(image_path)

        results = predictor(
            text=["food"]
        )

        if not results or results[0].masks is None:
            raise RuntimeError(
                f"SAM 3 could not segment {food['name']}."
            )

        raw_masks = (
            results[0].masks.data
            .detach()
            .cpu()
            .numpy()
        )

        mask = self._select_mask_for_box(
            masks=raw_masks,
            bbox=(
                food["x1"],
                food["y1"],
                food["x2"],
                food["y2"],
            ),
            image_shape=image_shape,
        )

        return mask

    def calculate_food_area(
        self,
        food_mask,
        scale_map,
    ):
        # Each pixel represents approximately:
        # (cm/pixel)^2 square centimetres.
        valid_pixels = (
            food_mask
            & np.isfinite(scale_map)
            & (scale_map > 0)
        )

        pixel_count = int(valid_pixels.sum())

        if pixel_count == 0:
            raise RuntimeError(
                "No food-mask pixels have valid depth."
            )

        pixel_area_map = np.square(scale_map)

        area_cm2 = float(
            np.sum(pixel_area_map[valid_pixels])
        )

        return {
            "area_pixels": pixel_count,
            "area_cm2": area_cm2,
            "food_mask": food_mask,
        }

    # ---------------------------------------------------------
    # RUN ALL SIX STAGES
    # ---------------------------------------------------------
    def estimate(self, image_path):
        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(
                f"Could not read image: {image_path}"
            )

        image_shape = image.shape

        # Stage 1
        foods = self.detect_food(image_path)

        # Stage 2
        utensil = self.detect_utensil(image_path)

        # Stage 3
        utensil_data = self.segment_utensil(
            image_path,
            utensil,
        )

        # Stage 4
        depth_map = self.get_depth_map(
            image_path,
            image_shape,
        )

        # Stage 5
        calibration = self.calculate_scale_map(
            depth_map=depth_map,
            utensil_mask=utensil_data["mask"],
            utensil_length_px=utensil_data["length_px"],
        )

        scale_map = calibration["scale_map"]

        # Stage 6
        food_results = []

        print("\n[STAGE 6] Calculating food areas...")

        for index, food in enumerate(foods):
            try:
                food_mask = self.segment_food_item(
                    image_path,
                    food,
                    image_shape,
                )

                measurements = self.calculate_food_area(
                    food_mask,
                    scale_map,
                )

                item = {
                    "food_type": food["name"],
                    "confidence": food["confidence"],
                    "bbox": (
                        food["x1"],
                        food["y1"],
                        food["x2"],
                        food["y2"],
                    ),
                    "food_mask": measurements["food_mask"],
                    "food_area_pixels": (
                        measurements["area_pixels"]
                    ),
                    "food_area_cm2": (
                        measurements["area_cm2"]
                    ),
                }

                food_results.append(item)

                print(
                    f"  {index + 1}. {food['name']}: "
                    f"{item['food_area_cm2']:.2f} cm² "
                    f"({item['food_area_pixels']} valid pixels)"
                )

            except RuntimeError as exc:
                print(
                    f"  Could not estimate {food['name']}: {exc}"
                )

        if not food_results:
            raise RuntimeError(
                "No food areas could be calculated."
            )

        return {
            "foods": food_results,
            "utensil": utensil,
            "utensil_mask": utensil_data["mask"],
            "utensil_length_px": utensil_data["length_px"],
            "assumed_utensil_length_cm": (
                self.UTENSIL_LENGTH_CM
            ),
            "depth_map": depth_map,
            "scale_map_cm_per_pixel": scale_map,
            "reference_depth": calibration["reference_depth"],
            "reference_cm_per_pixel": (
                calibration["reference_cm_per_pixel"]
            ),
        }


# =============================================================
# RUN
# =============================================================
if __name__ == "__main__":

    FOOD_MODEL_PATH = r"C:\Study\Major\Food\classif.pt"

    UTENSIL_MODEL_PATH = (
        r"C:\Study\Major\runs\detect\runs"
        r"\spoon_yolo11m\weights\last.pt"
    )

    SAM_MODEL_PATH = "sam3.pt"

    IMAGE_PATH = r"Spoon/Tests/Test3.jpeg"

    pipeline = FoodAreaPipeline(
        food_model_path=FOOD_MODEL_PATH,
        utensil_model_path=UTENSIL_MODEL_PATH,
        sam_model_path=SAM_MODEL_PATH,
        depth_api_url=(
            "http://localhost:8000/predict-depth-map"
        ),
    )

    output = pipeline.estimate(IMAGE_PATH)

    print("\n========== FINAL RESULTS ==========")

    for item in output["foods"]:
        print(
            f"{item['food_type']}: "
            f"{item['food_area_cm2']:.2f} cm²"
        )

ModuleNotFoundError: No module named 'Food_detection'